In [ ]:
# !pip install torch_geometric

In [ ]:
# !pip install torch-scatter -f https://data.pyg.org/whl/torch-2.5.1+cu121.html

In [ ]:
# import all libraries needed downstream
import os
import gc
import wandb
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
import scipy
from sklearn.metrics import mean_squared_error
import math
import networkx as nx
import seaborn as sns
import time
from torch.nn import Linear
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import degree
from torch_geometric.nn import ChebConv, GraphConv, GCNConv, TAGConv, GATConv
from torch_geometric.data import Data
from torch.utils.data import TensorDataset
from torch_geometric.loader import DataLoader
import torch_scatter
from typing import Dict, Tuple, List, Optional
import matplotlib.pyplot as plt
from torch_scatter import scatter_softmax, scatter_sum

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
!nvidia-smi

In [ ]:
def load_grid_data(grid_size:int):
   # Load the data
    data = np.load(f'/home/oarowolo/workfile/OPFData/{grid_size}bus_combined_dataset.npz') 

    # Get all keys
    print("Available keys in the dataset:")
    for key in data.files:
        # Print the key and its array shape
        print(f"{key}: shape {data[key].shape}") 
        
        
        
        # Access grid input features
        grid_bus = data['grid_bus']  
        grid_generator = data['grid_generator']
        grid_load = data['grid_load']
        grid_shunt = data['grid_shunt']
        grid_ac_line_features = data['grid_ac_line_features']
        grid_transformer_features = data['grid_transformer_features']
        grid_ac_line_receivers = data['grid_ac_line_receivers']
        grid_ac_line_senders = data['grid_ac_line_senders']
        grid_transformer_senders = data['grid_transformer_senders']
        grid_transformer_receivers = data['grid_transformer_receivers']
        
        solution_bus = data['solution_bus']  
        solution_generator = data['solution_generator'] 
        solution_objective = data['metadata_objective']
        
        solution_objective = solution_objective.reshape(-1,1)
            
        grid_generator_link_receivers = data['grid_generator_link_receivers']
        grid_load_link_receivers = data['grid_load_link_receivers']
        grid_shunt_link_receivers = data['grid_shunt_link_receivers']
        
        generator_indices = grid_generator_link_receivers[0]
        load_indices = grid_load_link_receivers[0]
        shunt_indices = grid_shunt_link_receivers[0]
        grid_transformer_senders = grid_transformer_senders[0]
        grid_transformer_receivers = grid_transformer_receivers[0]
        grid_ac_line_senders  = grid_ac_line_senders[0]
        grid_ac_line_receivers = grid_ac_line_receivers[0]
        
        return grid_bus, grid_generator, grid_load, grid_shunt, grid_ac_line_features,grid_ac_line_senders,grid_ac_line_receivers, grid_transformer_features, grid_transformer_senders, grid_transformer_receivers, solution_bus, solution_generator, solution_objective, generator_indices, load_indices, shunt_indices

In [ ]:
system_size = 2000

In [ ]:
grid_bus, grid_generator, grid_load, grid_shunt, grid_ac_line_features,grid_ac_line_senders,grid_ac_line_receivers, grid_transformer_features, grid_transformer_senders, grid_transformer_receivers,solution_bus, solution_generator, solution_objective, generator_indices,load_indices, shunt_indices = load_grid_data(system_size)

In [ ]:
grid_ac_line_senders = grid_ac_line_senders.reshape(-1,1)
grid_ac_line_receivers = grid_ac_line_receivers.reshape(-1,1)
grid_transformer_senders = grid_transformer_senders.reshape(-1,1)
grid_transformer_receivers = grid_transformer_receivers.reshape(-1,1)

In [ ]:
generator_indices.shape

In [ ]:
branch_list = list(zip(grid_ac_line_senders.flatten(), grid_ac_line_receivers.flatten()))
transformer_list = list(zip(grid_transformer_senders.flatten(), grid_transformer_receivers.flatten()))
for k in transformer_list:
    branch_list.append(k)

In [ ]:
edge_inputs = np.zeros((len(branch_list),11))

edge_inputs[:grid_ac_line_features.shape[1],:9] = grid_ac_line_features[0]  # rearranging edge inputs to align for transformers and transmission lines
edge_inputs[grid_ac_line_features.shape[1]:,:2] =  grid_transformer_features[0,:,:2]
edge_inputs[grid_ac_line_features.shape[1]:,2:4] =  grid_transformer_features[0,:,9:]
edge_inputs[grid_ac_line_features.shape[1]:,4:9] =  grid_transformer_features[0,:,2:7]
edge_inputs[grid_ac_line_features.shape[1]:,9:] =  grid_transformer_features[0,:,7:9]
edge_inputs[:grid_ac_line_features.shape[1],9:10] = 1.0

In [ ]:
def compute_gandb(edge_iputs):

    line_r = edge_inputs[:,4:5]
    line_x = edge_inputs[:,5:6]

    line_g = line_r/(line_r**2 + line_x**2)
    line_b = -line_x/(line_r**2 + line_x**2)

    return line_g, line_b


In [ ]:
edge_g, edge_b = compute_gandb(edge_inputs)

In [ ]:
def get_B_matrix(N, edges, edge_weights):
    # Create a zero tensor of shape (N,N)
    B_matrix = torch.zeros((N, N)).to(torch.float64)
    
    # Unpack the edges into source and destination nodes
    sources, destinations = zip(*edges)
    
    # Use advanced indexing to place weights in the right spots
    B_matrix[sources, destinations] = edge_weights.squeeze()
    B_matrix[destinations, sources] = edge_weights.squeeze()
    return B_matrix

In [ ]:
def adjacency_to_laplacian(B_adj):

    # Ensure matrix is square
    assert B_adj.shape[0] == B_adj.shape[1], "Input must be square"

    # Copy to avoid modifying original
    B_laplacian = B_adj.copy()

    # Set diagonal as row sum of adjacency (i.e., degree)
    np.fill_diagonal(B_laplacian, -B_adj.sum(axis=1))

    return B_laplacian


In [ ]:
B_weighted = get_B_matrix(system_size, branch_list,torch.tensor(edge_b))

In [ ]:
import scipy.sparse as sp
import scipy.sparse.linalg as spla

def effective_resistance_matrix(b_mat):
    """
    Computes the effective resistance matrix from a susceptance adjacency matrix.
    Converts it into a Laplacian first.
    
    Parameters:
    b_mat (ndarray): weighted laplacian matrix
    
    Returns:
    ndarray: Effective resistance matrix (N x N)
    """
    # Ensure symmetry
    L = b_mat

    n = L.shape[0]

    # Remove reference node (last row and column) to deal with singularity
    keep = np.arange(n - 1)
    L_reduced = L[np.ix_(keep, keep)]

    # Invert reduced Laplacian
    L_reduced_inv = np.linalg.inv(L_reduced)

    # Expand to full pseudoinverse
    L_plus = np.zeros((n, n))
    L_plus[np.ix_(keep, keep)] = L_reduced_inv

    # Project to orthogonal component (to make it true pseudoinverse)
    I = np.eye(n)
    ones = np.ones((n, n)) / n
    L_plus = (I - ones) @ L_plus @ (I - ones)

    # Compute resistance: R_ij = L^+_ii + L^+_jj - 2L^+_ij
    diag = np.diag(L_plus)
    R = diag[:, None] + diag[None, :] - 2 * L_plus
    return R

In [ ]:
b_mat = np.array(B_weighted)
B_lap = adjacency_to_laplacian(b_mat)
e_R = effective_resistance_matrix(B_lap)
e_R_norm = e_R/e_R.max()

In [ ]:
# Vectorized version (more efficient for large matrices)
def compute_row_statistics_vectorized(resistance_matrix):
    """
    Vectorized computation of row statistics excluding diagonal elements.
    
    Parameters:
    resistance_matrix: numpy array of shape (N, N) with values between 0 and 1
    
    Returns:
    numpy array of shape (N, 5) with columns: [mean, median, std, max, min]
    """
    N = resistance_matrix.shape[0]
    
    # Create a mask to exclude diagonal elements
    mask = ~np.eye(N, dtype=bool)
    
    # Initialize result matrix
    stats_matrix = np.zeros((N, 5))
    
    # For each row, extract non-diagonal elements and compute statistics
    for i in range(N):
        row_no_diag = resistance_matrix[i, mask[i]]
        
        stats_matrix[i, 0] = np.mean(row_no_diag)
        stats_matrix[i, 1] = np.median(row_no_diag)
        stats_matrix[i, 2] = np.std(row_no_diag)
        stats_matrix[i, 3] = np.max(row_no_diag)
        stats_matrix[i, 4] = np.min(row_no_diag)
    
    return stats_matrix

In [ ]:
raw_PE = compute_row_statistics_vectorized(e_R_norm)
bus_pe = torch.tensor(raw_PE,dtype=torch.float)

In [ ]:
batch_size = 16

In [ ]:
### This is the part to load and insert N-1 data for loads and also for solutions

In [ ]:
# Load the data
data_n1 = np.load(f'/home/oarowolo/workfile/OPFData/{system_size}bus_nminusone_combined_dataset.npz',allow_pickle=True)

# Get all keys
print("Available keys in the dataset:")
for key in data_n1.files:
    # Print the key and its array shape
    print(f"{key}: shape {data_n1[key].shape}")

In [ ]:
grid_bus_n1 = data_n1['grid_bus']  
grid_generator_n1 = data_n1['grid_generator']
grid_load_n1 = data_n1['grid_load']  #### we need this
grid_shunt_n1 = data_n1['grid_shunt']
grid_ac_line_features_n1 = data_n1['grid_ac_line_features']
grid_transformer_features_n1 = data_n1['grid_transformer_features']
grid_ac_line_receivers_n1 = data_n1['grid_ac_line_receivers']
grid_ac_line_senders_n1 = data_n1['grid_ac_line_senders']
grid_transformer_senders_n1 = data_n1['grid_transformer_senders']
grid_transformer_receivers_n1 = data_n1['grid_transformer_receivers']
grid_generator_link_senders_n1 = data_n1['grid_generator_link_senders']
grid_generator_link_receivers_n1 = data_n1['grid_generator_link_receivers']
solution_bus_n1 = data_n1['solution_bus']  ### we need this
solution_generator_n1 = data_n1['solution_generator'] ### we need this
solution_objective_n1 = data_n1['metadata_objective']
solution_objective_n1 = solution_objective_n1.reshape(-1,1)  ## we need this
grid_load_link_receivers_n1 = data_n1['grid_load_link_receivers']
grid_load_link_senders_n1 = data_n1['grid_load_link_senders']
grid_shunt_link_receivers_n1 = data_n1['grid_shunt_link_receivers']
grid_shunt_link_senders_n1 = data_n1['grid_shunt_link_senders']

In [ ]:
solution_generator_new = np.zeros_like(solution_generator)

In [ ]:
def build_mapping(universe):
    """
    Build mapping dict from labels in universe to compact indices 0..len(universe)-1.
    """
    return {lab: i for i, lab in enumerate(universe)}

def compact_with_fixed_map(labels, mapping):
    """
    Map each label to its fixed compact index using pre-built mapping.
    """
    return np.array([mapping[lab] for lab in labels])

def fill_compacted_loop(arr1, arr2, arr3, universe):
    mapping = build_mapping(universe)
    N = arr1.shape[0]

    for i in range(N):
        labels = np.asarray(arr3[i])
        values = np.asarray(arr2[i])
        if labels.size == 0:
            continue
        compact = np.array([mapping[lab] for lab in labels])
        arr1[i, compact, :] = values
    return arr1

In [ ]:
solution_generator_filled = fill_compacted_loop(solution_generator_new, solution_generator_n1, grid_generator_link_receivers_n1, generator_indices)

In [ ]:
grid_load_n1 = rr = np.array(grid_load_n1.tolist(), dtype=np.float32)

In [ ]:
solution_objective_n1 = np.array(solution_objective_n1.tolist(), dtype=np.float32)

In [ ]:
solution_bus_n1 = np.array(solution_bus_n1.tolist(), dtype=np.float32)

In [ ]:
## wandb set-up
api_key = ''   ####specify your api key
wandb.login(key=api_key)

In [ ]:
run = wandb.init(
      # Set the project where this run will be logged
      project="Towards_Generalization_of_GNN_for_ACOPF",
      # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
      name=f"Naive_baseline_{system_size}_HybridHeteroGNN_5_256_PQVT_top40",
      # Track hyperparameters and run metadata
      config={
      "architecture": "GNN",
      "dataset": "Full",
      "epochs": 100,
      })

In [ ]:
solution_objective_n1.mean()

In [ ]:
sorted_indices = np.argsort(solution_objective_n1.flatten())[::-1]

# Take the top 1000
top_1000_indices = sorted_indices[:1000]

# print(top_1000_indices)

In [ ]:
import torch
from torch_geometric.data import HeteroData

def create_grid_hetero_data(
    grid_bus,                    # (300000, 14, 4)
    grid_generator,              # (300000, 5, 11)
    grid_load,                   # (300000, 11, 2)
    grid_shunt,                  # (300000, 1, 2)
    grid_ac_line_features,       # (300000, 17, 9)
    grid_transformer_features,   # (300000, 3, 11)
    grid_ac_line_senders,        # (17, 1)
    grid_ac_line_receivers,      # (17, 1)
    grid_transformer_senders,    # (3, 1)
    grid_transformer_receivers,  # (3, 1)
    generator_indices,           # Indices connecting generators to buses
    load_indices,                # Indices connecting loads to buses
    shunt_indices,               # Indices connecting shunts to buses
    solution_bus,                # (300000, 14, 2) - solutions for bus nodes
    solution_generator,          # (300000, 5, 2) - solutions for generator nodes
    batch_idx=0                  # Batch index to extract (or None for all batches)
):
    
    # Create a new HeteroData instance
    data = HeteroData()
    
    # Process a single batch if specified, otherwise we'd need to handle batching differently
    if batch_idx is not None:
        # Extract features for the specified batch
        bus_features = torch.tensor(grid_bus[batch_idx], dtype=torch.float)
        generator_features = torch.tensor(grid_generator[batch_idx], dtype=torch.float)
        load_features = torch.tensor(grid_load[batch_idx], dtype=torch.float)
        shunt_features = torch.tensor(grid_shunt[batch_idx], dtype=torch.float)
        
        ac_line_features = torch.tensor(grid_ac_line_features[batch_idx], dtype=torch.float)
        transformer_features = torch.tensor(grid_transformer_features[batch_idx], dtype=torch.float)
        
        # Extract solution values for the specified batch
        bus_solutions = torch.tensor(solution_bus[batch_idx], dtype=torch.float)
        generator_solutions = torch.tensor(solution_generator[batch_idx], dtype=torch.float)
        
        # Add node features
        data['bus'].x = bus_features
        data['generator'].x = generator_features
        data['load'].x = load_features
        data['shunt'].x = shunt_features

        #Add node positional encoding
        data['bus'].pe = bus_pe
        data['generator'].pe = bus_pe[generator_indices]  ## added arbitrary values to distinguish buses from special nodes
        data['load'].pe = bus_pe[load_indices]
        data['shunt'].pe = bus_pe[shunt_indices]

        # Add solution values as target values (y)
        data['bus'].y = bus_solutions
        data['generator'].y = generator_solutions
        
        # Add edge indices and features for AC lines (bus to bus)
        senders = torch.tensor(grid_ac_line_senders.flatten(), dtype=torch.long)
        receivers = torch.tensor(grid_ac_line_receivers.flatten(), dtype=torch.long)
        edge_index = torch.stack([senders, receivers], dim=0)
        data['bus', 'ac_line', 'bus'].edge_index = edge_index
        data['bus', 'ac_line', 'bus'].edge_attr = ac_line_features
        
        # Add edge indices and features for transformers (bus to bus)
        senders = torch.tensor(grid_transformer_senders.flatten(), dtype=torch.long)
        receivers = torch.tensor(grid_transformer_receivers.flatten(), dtype=torch.long)
        edge_index = torch.stack([senders, receivers], dim=0)
        data['bus', 'transformer', 'bus'].edge_index = edge_index
        data['bus', 'transformer', 'bus'].edge_attr = transformer_features

        # for now, let's assume the pseudoedges are not bidirectional, they only influence the buses they are attached to (destination node), we cannot update their latent representation
        
        # Add pseudo-edges from generators to buses
        gen_to_bus = torch.tensor(generator_indices.flatten(), dtype=torch.long)
        N = system_size
        gen_index = torch.arange(len(gen_to_bus), dtype=torch.long)  
        gen_edge_index = torch.stack([gen_index, gen_to_bus], dim=0)
        data['generator', 'connects_to', 'bus'].edge_index = gen_edge_index
        data['generator', 'connects_to', 'bus'].edge_attr = torch.ones((len(gen_to_bus), 3))
        
        # Add pseudo-edges from loads to buses
        load_to_bus = torch.tensor(load_indices.flatten(), dtype=torch.long)
        load_index = torch.arange(len(load_to_bus), dtype=torch.long)
        load_edge_index = torch.stack([load_index, load_to_bus], dim=0)
        data['load', 'connects_to', 'bus'].edge_index = load_edge_index
        data['load', 'connects_to', 'bus'].edge_attr = torch.ones((len(load_to_bus), 3))
        
        # Add pseudo-edges from shunts to buses
        shunt_to_bus = torch.tensor(shunt_indices.flatten(), dtype=torch.long)
        shunt_index = torch.arange(len(shunt_to_bus), dtype=torch.long)
        shunt_edge_index = torch.stack([shunt_index, shunt_to_bus], dim=0)
        data['shunt', 'connects_to', 'bus'].edge_index = shunt_edge_index
        data['shunt', 'connects_to', 'bus'].edge_attr = torch.ones((len(shunt_to_bus), 3))
    
    else:
        raise NotImplementedError("Processing all batches at once is not implemented in this example")
    
    return data



def create_dataloader(
    grid_bus,
    grid_generator,
    grid_load,
    grid_shunt,
    grid_ac_line_features,
    grid_transformer_features,
    grid_ac_line_senders,
    grid_ac_line_receivers,
    grid_transformer_senders,
    grid_transformer_receivers,
    generator_indices,
    load_indices,
    shunt_indices,
    solution_bus,
    solution_generator,
    batch_size=batch_size,
    shuffle = True
):
    """
    Create a dataloader for the heterogeneous graph data with solutions.
    """
    from torch_geometric.loader import DataLoader
    
    # Create a list of HeteroData objects
    dataset = []
    
    for i in range(len(grid_bus)):  # Process up to 1000 samples for this example
        data = create_grid_hetero_data(
            grid_bus, 
            grid_generator,
            grid_load,
            grid_shunt,
            grid_ac_line_features,
            grid_transformer_features,
            grid_ac_line_senders,
            grid_ac_line_receivers,
            grid_transformer_senders,
            grid_transformer_receivers,
            generator_indices,
            load_indices,
            shunt_indices,
            solution_bus,
            solution_generator,
            batch_idx=i
        )
        dataset.append(data)
    
    # Create a DataLoader
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,num_workers=min(8, torch.get_num_threads()))
    
    return loader

In [ ]:
def train_val_test_split(train_ratio=0.9, val_ratio=0.05, test_ratio=0.05, seed=42):

    # Set random seed for reproducibility
    torch.manual_seed(seed)

    # Shuffle indices
    num_samples = 300000
    indices = torch.randperm(num_samples)

    # Compute split sizes
    train_size = int(train_ratio * num_samples)
    val_size = int(val_ratio * num_samples)

    # Split indices
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    return train_indices, val_indices, test_indices

In [ ]:
train_indices, val_indices, test_indices = train_val_test_split()

In [ ]:
test_indices_arr = np.array(test_indices)

In [ ]:
test_indices_arr

In [ ]:
test_intersection =  np.intersect1d(test_indices_arr, top_1000_indices)

print(len(test_intersection))

In [ ]:
test_intersection = torch.tensor(test_intersection)

In [ ]:
test_intersection = test_intersection[:40]

In [ ]:
len(test_intersection)

In [ ]:
### calculate means

In [ ]:
solution_objective[train_indices].mean()

In [ ]:
solution_objective_n1[train_indices].mean()

In [ ]:
solution_objective_n1[test_intersection].mean()

In [ ]:
train_loader = create_dataloader(grid_bus[train_indices],
                                grid_generator[train_indices],
                                grid_load_n1[train_indices], ## replaced here
                                grid_shunt[train_indices],
                                grid_ac_line_features[train_indices],
                                grid_transformer_features[train_indices],
                                grid_ac_line_senders,
                                grid_ac_line_receivers,
                                grid_transformer_senders,
                                grid_transformer_receivers,
                                generator_indices,
                                load_indices,
                                shunt_indices,
                                solution_bus_n1[train_indices], ## replaced here
                                solution_generator_filled[train_indices], ## replaced here
                                batch_size=batch_size,
                                shuffle = True)

In [ ]:
val_loader = create_dataloader(grid_bus[val_indices],
                                grid_generator[val_indices],
                                grid_load_n1[val_indices], ## replaced here
                                grid_shunt[val_indices],
                                grid_ac_line_features[val_indices],
                                grid_transformer_features[val_indices],
                                grid_ac_line_senders,
                                grid_ac_line_receivers,
                                grid_transformer_senders,
                                grid_transformer_receivers,
                                generator_indices,
                                load_indices,
                                shunt_indices,
                                solution_bus_n1[val_indices], ## replaced here
                                solution_generator_filled[val_indices], ## replaced here
                                batch_size=batch_size,
                                shuffle = False)

In [ ]:
# test_loader = create_dataloader(grid_bus[test_indices],
#                                 grid_generator[test_indices],
#                                 grid_load_n1[test_indices], ## replaced here
#                                 grid_shunt[test_indices],
#                                 grid_ac_line_features[test_indices],
#                                 grid_transformer_features[test_indices],
#                                 grid_ac_line_senders,
#                                 grid_ac_line_receivers,
#                                 grid_transformer_senders,
#                                 grid_transformer_receivers,
#                                 generator_indices,
#                                 load_indices,
#                                 shunt_indices,
#                                 solution_bus_n1[test_indices], ## replaced here
#                                 solution_generator_filled[test_indices], ## replaced here
#                                 batch_size=batch_size,
#                                 shuffle = False)

In [ ]:
test_loader = create_dataloader(grid_bus[test_intersection],
                                grid_generator[test_intersection],
                                grid_load_n1[test_intersection], ## replaced here
                                grid_shunt[test_intersection],
                                grid_ac_line_features[test_intersection],
                                grid_transformer_features[test_intersection],
                                grid_ac_line_senders,
                                grid_ac_line_receivers,
                                grid_transformer_senders,
                                grid_transformer_receivers,
                                generator_indices,
                                load_indices,
                                shunt_indices,
                                solution_bus_n1[test_intersection], ## replaced here
                                solution_generator_filled[test_intersection], ## replaced here
                                batch_size=batch_size,
                                shuffle = False)

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size, layers, layernorm=True, use_leaky=False): 
        super().__init__()
        # Use Sequential instead of ModuleList for faster forward pass
        modules = []
        for i in range(layers):
            modules.append(torch.nn.Linear(
                input_size if i == 0 else hidden_size,
                output_size if i == layers - 1 else hidden_size,
            ))
            if i != layers - 1:
                modules.append(torch.nn.ReLU())
            if use_leaky:
                modules.append(torch.nn.LeakyReLU(negative_slope=0.02))
        if layernorm:
            modules.append(torch.nn.LayerNorm(output_size))
        
        self.network = torch.nn.Sequential(*modules)

    def forward(self, x):
        # Sequential is faster than iterating through ModuleList
        return self.network(x)

In [ ]:
# from performer_pytorch import SelfAttention
from torch_geometric.utils import to_dense_batch
from torch_geometric.nn.attention import PerformerAttention

class HeteroPerformerLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads=1, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.attn = PerformerAttention(
            channels=hidden_dim,
            heads=num_heads
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        # Post-attention MLP
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
    def forward(self, x_dict, xm_dict, batch_dict):
            # Flatten all node types
            flat_x, flat_xm, flat_batch = [], [], []
            slices = {}
            offset = 0
    
            for ntype in x_dict:
                x = x_dict[ntype]
                xm = xm_dict[ntype]
                b = batch_dict[ntype]  # batch indices for each node
    
                slices[ntype] = slice(offset, offset + x.size(0))
                flat_x.append(x)
                flat_xm.append(xm)
                flat_batch.append(b)
                offset += x.size(0)
    
            x_all = torch.cat(flat_x, dim=0)         # [N, D]
            xm_all = torch.cat(flat_xm, dim=0)       # [N, D]
            batch_all = torch.cat(flat_batch, dim=0) # [N]           
    
            # Convert to [B, N_max, D] and mask
            x_dense, mask = to_dense_batch(x_all, batch_all)   # [B, N, D], [B, N]
            
            # Apply masked Performer attention
            x_attn = self.attn(x_dense, mask=mask)              # [B, N, D]
    
            # Residual + Norm
            xt_out = self.norm1(x_dense + x_attn)

            # Unpad: [real_nodes, D]
            xt_out = xt_out[mask]
    
            x_comb = self.norm2(xt_out + xm_all)
            x_final = self.mlp(x_comb)

            # Unflatten by slice
            return {ntype: x_final[slices[ntype]] for ntype in x_dict.keys()}


In [ ]:
class HeteroInteractionNetwork(nn.Module):
    def __init__(self, node_types, edge_types,physical_edge_types, hidden_size, layers):
        super().__init__()
        
        self.physical_edge_types = physical_edge_types
        self.edge_updaters = nn.ModuleDict()
        for src, rel, dst in edge_types:
            edge_key = f"{src}_{rel}_{dst}"
            edge_type = (src, rel, dst)
            
            # For physical edges (ac_line and transformer), use node + edge features
            if edge_type in physical_edge_types:
                self.edge_updaters[edge_key] = MLP(hidden_size * 3, hidden_size, hidden_size, layers)
            else:
                # For other edge types, only use node features
                self.edge_updaters[edge_key] = MLP(hidden_size * 2, hidden_size, hidden_size, layers)
        
        # Create a node updater for each node type
        self.node_updaters = nn.ModuleDict({
            node_type: MLP(hidden_size * 2, hidden_size, hidden_size, layers)
            for node_type in node_types
        })

    
    def forward(self, x_dict, edge_indices_dict, edge_features_dict):
        # Store updated node and edge features
        updated_edge_features = {}
        
        # Prepare aggregated messages storage
        aggregated_messages = {node_type: torch.zeros_like(feat) 
                              for node_type, feat in x_dict.items()}
        
        # Process each edge type in parallel
        for edge_type, edge_index in edge_indices_dict.items():
            src_type, rel_type, dst_type = edge_type
            edge_key = f"{src_type}_{rel_type}_{dst_type}"
            
            # Get node features for this edge
            src, dst = edge_index
            x_i = x_dict[dst_type][dst]  # Destination nodes
            x_j = x_dict[src_type][src]  # Source nodes
            # Update edge features based on edge type
            if edge_type in self.physical_edge_types:
                # For physical edges, include edge features in message
                edge_feature = edge_features_dict[edge_type]
                edge_msg = torch.cat((x_i, x_j, edge_feature), dim=-1)
                updated_edge = self.edge_updaters[edge_key](edge_msg)
                updated_edge_features[edge_type] =  updated_edge + edge_feature  
            else:
                # For non-physical edges, only use node features
                edge_msg = torch.cat((x_i, x_j), dim=-1)
                updated_edge = self.edge_updaters[edge_key](edge_msg)
                
                #Check if we have existing edge features from previous layers
                # if edge_type in edge_features_dict:
                #     edge_feature = edge_features_dict[edge_type]
                #     updated_edge_features[edge_type] = edge_feature + updated_edge  ### removed residual connection for non-physical edges here
                # else:
                    # First layer - initialize with the computed edge features
                updated_edge_features[edge_type] = updated_edge 
            
            # Efficient message aggregation using torch_scatter
            aggregated_messages[dst_type] = torch_scatter.scatter_add(updated_edge, dst, dim=0, out=aggregated_messages[dst_type])
            aggregated_messages[src_type] = torch_scatter.scatter_add(updated_edge, src, dim=0, out=aggregated_messages[src_type])
        
        # Update node features
        updated_nodes = {}
        for node_type, x in x_dict.items():
            # Combine node features with aggregated messages
            node_input = torch.cat((x, aggregated_messages[node_type]), dim=-1)
            node_update = self.node_updaters[node_type](node_input)
            updated_nodes[node_type] = x + node_update 
        
        return updated_nodes, updated_edge_features


In [ ]:
class HeteroInteractGNN(torch.nn.Module):
    def __init__(
        self,
        hidden_size=256,
        n_mp_layers=5,
        bus_features=4,
        gen_features=11,
        load_features=2,
        shunt_features=2,
        ac_line_features=9,
        transformer_features=11,
        connects_to_features=3,
        output_dim=2
    ):
        super().__init__()
        
        # Define node and edge types
        self.node_types = ['bus', 'generator', 'load', 'shunt']
        self.edge_types = [
            ('bus', 'ac_line', 'bus'),
            ('bus', 'transformer', 'bus'),
            ('generator', 'connects_to', 'bus'),
            ('load', 'connects_to', 'bus'),
            ('shunt', 'connects_to', 'bus')
        ]

        self.physical_edge_types = [
            ('bus', 'ac_line', 'bus'),
            ('bus', 'transformer', 'bus')
        ]        
        #Node encoders - separate MLP for each node type
        self.node_encoders = nn.ModuleDict({
            'bus': MLP(bus_features, hidden_size, hidden_size-5, 2),
            'generator': MLP(gen_features, hidden_size, hidden_size-5, 2),
            'load': MLP(load_features, hidden_size, hidden_size-5, 2),
            'shunt': MLP(shunt_features, hidden_size, hidden_size-5, 2)
        })


        self.global_attn_layers = nn.ModuleList([
            HeteroPerformerLayer(hidden_size)
            for _ in range(n_mp_layers)
        ])

        #Edge encoders - separate MLP for each edge type
        self.edge_encoders = nn.ModuleDict({
            'ac_line': MLP(ac_line_features, hidden_size, hidden_size, 2),
            'transformer': MLP(transformer_features, hidden_size, hidden_size, 2)
        })
        
        # Interaction network layers
        self.n_mp_layers = n_mp_layers
        self.layers = torch.nn.ModuleList([
            HeteroInteractionNetwork(self.node_types, self.edge_types,self.physical_edge_types, hidden_size, 2)
            for _ in range(n_mp_layers)
        ])

        
        # Node decoders - separate for bus and generator
        self.node_decoders = nn.ModuleDict({
            'bus': MLP(hidden_size, hidden_size, output_dim, 2, layernorm=False),
            'generator': MLP(hidden_size, hidden_size, output_dim, 2,layernorm=False)
        })

    def forward(self, data):
        # Encode node features
        x_dict_init = {}
        x_dict = {}
        
        # Batch node encoding
        for node_type in self.node_types:
            if hasattr(data[node_type], 'x'):
                x_dict_init[node_type] = self.node_encoders[node_type](data[node_type].x)
                x_dict[node_type] = torch.cat([x_dict_init[node_type], data[node_type].pe], dim=-1)

        # Encode edge features
        edge_feature_dict = {}
        for src, rel, dst in self.edge_types:
            edge_type = (src, rel, dst)
            if (edge_type in self.physical_edge_types and 
                edge_type in data.edge_types and 
                hasattr(data[edge_type], 'edge_attr')):
                edge_feature_dict[edge_type] = self.edge_encoders[rel](data[edge_type].edge_attr)
        
        # Extract edge indices
        edge_index_dict = {
            edge_type: data[edge_type].edge_index
            for edge_type in self.edge_types
            if edge_type in data.edge_types and hasattr(data[edge_type], 'edge_index')
        }

        batch_dict = {
            ntype: data[ntype].batch
            for ntype in x_dict
        }
        
        # Apply message passing layers
        for i in range(self.n_mp_layers):
            xm_dict, edge_feature_dict = self.layers[i](x_dict, edge_index_dict, edge_feature_dict)
            x_dict = self.global_attn_layers[i](x_dict, xm_dict, batch_dict)

        # Apply decoders
        output = {}
        output['bus'] = torch.sigmoid(self.node_decoders['bus'](x_dict['bus']))
        output['generator'] = torch.sigmoid(self.node_decoders['generator'](x_dict['generator']))
        
        return output

In [ ]:
def convert_voltage_bounds(model_input):

    num_nodes = model_input.shape[0]

    vmin = model_input[:,2:3]

    vmax = model_input[:,3:4]

    thetamin =  torch.tensor([-2.00]).to(device)
    thetamin = thetamin.tile((num_nodes,1))

    thetamax =  torch.tensor([2.00]).to(device)
    thetamax = thetamax.tile((num_nodes,1))

    bounds_up = torch.concat((thetamax, vmax), dim=1)
    bounds_down = torch.concat((thetamin, vmin),dim=1)


    return bounds_up, bounds_down
    

In [ ]:
def convert_power_bounds(model_input):

    num_nodes = model_input.shape[0]

    pmin = model_input[:,2:3]

    pmax = model_input[:,3:4]

    
    qmin = model_input[:,5:6]

    qmax = model_input[:,6:7]

    bounds_up = torch.concat((pmax, qmax), dim=1)
    bounds_down = torch.concat((pmin, qmin),dim=1)


    return bounds_up, bounds_down
    

In [ ]:
# Example training step
def train_model(model, trainloader, optimizer):
    
    model.train()
    total_loss = 0
    criterion = nn.MSELoss()
    
    for batch in trainloader:
    
        optimizer.zero_grad()
        
        batch = batch.to(device)

        # Forward pass
        pred_dict = model(batch)

        voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
        voltage_up = voltage_up.to(device)
        voltage_down = voltage_down.to(device)
        voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
        voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)

        power_up, power_down = convert_power_bounds(batch['generator'].x)
        power_up = power_up.to(device)
        power_down = power_down.to(device)
        powers = pred_dict['generator'] * (power_up - power_down) + power_down
        powers = torch.clamp(powers,min=power_down, max=power_up)
        

        combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
        combined_outputs = torch.cat([voltages, powers], dim=0)
        loss = criterion(combined_targets, combined_outputs)

        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(trainloader)

In [ ]:
# Example training step
def validate_model(model, val_loader):
    
    model.eval()
    total_loss = 0
    criterion = nn.MSELoss()
    
    for batch in val_loader:
        
        batch = batch.to(device)

        # Forward pass
        pred_dict = model(batch)

        voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
        voltage_up = voltage_up.to(device)
        voltage_down = voltage_down.to(device)
        voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
        voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)

        power_up, power_down = convert_power_bounds(batch['generator'].x)
        power_up = power_up.to(device)
        power_down = power_down.to(device)
        powers = pred_dict['generator'] * (power_up - power_down) + power_down
        powers = torch.clamp(powers,min=power_down, max=power_up)

        combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
        combined_outputs = torch.cat([voltages, powers], dim=0)

        loss = criterion(combined_targets, combined_outputs)
     
        total_loss += loss.item()
        
    return total_loss / len(val_loader)

In [ ]:
@torch.no_grad()
def test_model(model, testloader):

    model.eval()
    criterion = nn.MSELoss()
    
    total_loss = 0.0
    voltage_predictions = []
    voltage_targets = []
    power_predictions = []
    power_targets = []
    
    for batch in testloader:

        batch = batch.to(device)

        # Forward pass
        pred_dict = model(batch)

        voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
        voltage_up = voltage_up.to(device)
        voltage_down = voltage_down.to(device)
        voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
        voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)

        power_up, power_down = convert_power_bounds(batch['generator'].x)
        power_up = power_up.to(device)
        power_down = power_down.to(device)
        powers = pred_dict['generator'] * (power_up - power_down) + power_down
        powers = torch.clamp(powers,min=power_down, max=power_up)
        
        combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
        combined_outputs = torch.cat([voltages, powers], dim=0)
        loss = criterion(combined_targets, combined_outputs)
        
        total_loss += loss.item()

        # Store predictions and targets for overall metrics
        voltage_predictions.append(voltages.cpu())
        voltage_targets.append(batch['bus'].y.cpu())

        power_predictions.append(powers.cpu())
        power_targets.append(batch['generator'].y.cpu())

    
    return total_loss / len(testloader), voltage_predictions, voltage_targets, power_predictions, power_targets

In [ ]:
model = HeteroInteractGNN().to(device)

In [ ]:
tot_params = 0
for parameter in model.parameters():
  layer_ws = 1
  for val in parameter.shape:
      layer_ws*=val
  tot_params += layer_ws
print(f"Total number of parameters = {tot_params}")

In [ ]:
model.load_state_dict(torch.load(f"/home/oarowolo/workfile/OPFData/{system_size}_bus_Hybrid_HeteroGNN_5_256_PQVT.pth"))

In [ ]:
model.eval()

In [ ]:
# tot_params = 0
# for parameter in model.parameters():
#   layer_ws = 1
#   for val in parameter.shape:
#       layer_ws*=val
#   tot_params += layer_ws
# print(f"Total number of parameters = {tot_params}")

In [ ]:
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-5,weight_decay=5e-8)
# scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

In [ ]:
# training_losses = []
# validation_losses = []
# best_valid_loss = float('inf')
# early_stop_thresh = 100
# best_epoch = -1
# best_model_state = None
# num_epochs = 100

# for epoch in tqdm(range(num_epochs), desc="Training Progress"):
#     train_loss = train_model(model, train_loader, optimizer)
#     valid_loss = validate_model(model, val_loader)
#     training_losses.append(train_loss)
#     validation_losses.append(valid_loss)

#     wandb.log({"training_loss": train_loss, "validation_loss": valid_loss})

#     scheduler.step(train_loss)

#     if epoch % 10 == 0:
#       print(f'Epoch: {epoch}')
#       print(f'\tTrain Loss: {train_loss:.4f}')
#       print(f'\t Val. Loss: {valid_loss:.4f}')
#     if valid_loss < best_valid_loss:
#       best_valid_loss = valid_loss
#       best_model_state = deepcopy(model.state_dict())


# plt.subplots(figsize=(5,3))
# plt.plot([i for i in range(len(training_losses))], training_losses, 'r', label='Training loss')
# plt.plot([i for i in range(len(validation_losses))], validation_losses, 'g', label='Validation loss')
# plt.legend()
# plt.title(f'GNN Training and Validation loss',fontsize = 15)
# plt.xlabel('Epochs',fontsize = 12)
# plt.ylabel('MSE Loss',fontsize = 12)
# plt.semilogy()

# training_losses=np.array(training_losses)
# validation_losses=np.array(validation_losses)

# model.load_state_dict(best_model_state)
# model.eval()

In [ ]:
# torch.save(model.state_dict(), f"{system_size}_bus_Hybrid_HeteroGNN_5_256_PQVT.pth")
# wandb.save(f"{system_size}_bus_Hybrid_HeteroGNN_5_256_PQVT.pth")  # Upload to WandB

In [ ]:
test_loss, v_predictions, v_targets,p_predictions, p_targets = test_model(model, test_loader)

In [ ]:
print('loss on test data is ', test_loss)

In [ ]:
v_predictions = torch.cat(v_predictions, dim=0)
v_targets = torch.cat(v_targets, dim=0)

In [ ]:
p_predictions = torch.cat(p_predictions, dim=0)
p_targets = torch.cat(p_targets, dim=0)

In [ ]:
v_predictions = v_predictions.reshape(-1,system_size, 2)
v_targets = v_targets.reshape(-1, system_size, 2)

In [ ]:
p_predictions = p_predictions.reshape(-1,generator_indices.shape[0], 2)
p_targets = p_targets.reshape(-1, generator_indices.shape[0], 2)

In [ ]:
# compute voltage magnitude loss  and voltage angle loss separately
calc_loss = nn.MSELoss()
voltage_angle_loss = calc_loss(v_predictions[:,:,0],v_targets[:,:,0])
voltage_magnitude_loss = calc_loss(v_predictions[:,:,1],v_targets[:,:,1])

In [ ]:
print('average voltage angle discrepancy is  ', voltage_angle_loss)
print('average voltage magnitude discrepancy is  ', voltage_magnitude_loss)

In [ ]:
calc_loss = nn.MSELoss()
active_power_loss = calc_loss(p_predictions[:,:,0],p_targets[:,:,0])
reactive_power_loss = calc_loss(p_predictions[:,:,1],p_targets[:,:,1])
print('average gen active power error is  ', active_power_loss)
print('average gen reactive power error is  ', reactive_power_loss)

In [ ]:
load_demand = torch.zeros(grid_load[test_intersection].shape[0], system_size, grid_load[test_intersection].shape[-1])
load_demand[:,load_indices,:] = torch.tensor(grid_load[test_intersection])

In [ ]:
branch_list = list(zip(grid_ac_line_senders.flatten(), grid_ac_line_receivers.flatten()))
transformer_list = list(zip(grid_transformer_senders.flatten(), grid_transformer_receivers.flatten()))
for k in transformer_list:
    branch_list.append(k)

In [ ]:
edge_inputs = np.zeros((len(branch_list),11))

edge_inputs[:grid_ac_line_features.shape[1],:9] = grid_ac_line_features[0]  # rearranging edge inputs to align for transformers and transmission lines
edge_inputs[grid_ac_line_features.shape[1]:,:2] =  grid_transformer_features[0,:,:2]
edge_inputs[grid_ac_line_features.shape[1]:,2:4] =  grid_transformer_features[0,:,9:]
edge_inputs[grid_ac_line_features.shape[1]:,4:9] =  grid_transformer_features[0,:,2:7]
edge_inputs[grid_ac_line_features.shape[1]:,9:] =  grid_transformer_features[0,:,7:9]
edge_inputs[:grid_ac_line_features.shape[1],9:10] = 1.0

In [ ]:
def compute_gandb(edge_iputs):

    line_r = edge_inputs[:,4:5]
    line_x = edge_inputs[:,5:6]

    line_g = line_r/(line_r**2 + line_x**2)
    line_b = -line_x/(line_r**2 + line_x**2)

    return line_g, line_b


In [ ]:
edge_g, edge_b = compute_gandb(edge_inputs)

In [ ]:
edge_inputs = torch.tensor(edge_inputs)

In [ ]:
def calculate_only_branch_flows(
    demand: torch.Tensor,  # shape (batch_size,14,2) [real, imag]
    voltage: torch.Tensor,  # shape (batch_size,14,2) [real, imag]
    branches: list,  # list of 20 tuples (from_bus, to_bus)
    Yks: torch.Tensor,  # shape (14,2) [real, imag] shunt admittance
    Yij: torch.Tensor,  # shape (20,2) [real, imag] branch admittance
    Yijc: torch.Tensor,  # shape (20,2) [real, imag] branch charging admittance
    Tij: torch.Tensor,  # shape (20,2) [real, imag] transformation ratio
) -> torch.Tensor:
    batch_size = demand.shape[0]
    num_nodes = voltage.shape[1]
    
    # Helper function for batched complex multiplication
    def complex_mult_batch(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        return torch.stack([
            a[..., 0] * b[..., 0] - a[..., 1] * b[..., 1],
            a[..., 0] * b[..., 1] + a[..., 1] * b[..., 0]
        ], dim=-1)

    # Helper function for batched complex conjugate
    def complex_conj_batch(x: torch.Tensor) -> torch.Tensor:
        return torch.stack([x[..., 0], -x[..., 1]], dim=-1)

    # Helper function for batched complex division
    def complex_div_batch(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        denominator = b[..., 0]**2 + b[..., 1]**2
        return torch.stack([
            (a[..., 0] * b[..., 0] + a[..., 1] * b[..., 1]) / denominator,
            (a[..., 1] * b[..., 0] - a[..., 0] * b[..., 1]) / denominator
        ], dim=-1)

    # Initialize generator power tensor
    generator_power = torch.zeros_like(demand)
    
    # Calculate shunt power terms for each node (vectorized)
    v_mag_sq = torch.sum(voltage**2, dim=-1, keepdim=True)  # shape: (batch_size, 14, 1)
    v_mag_sq = torch.cat([v_mag_sq, torch.zeros_like(v_mag_sq)], dim=-1)  # shape: (batch_size, 14, 2)
    
    # Broadcast Yks to match batch dimension
    Yks_batch = Yks.expand(batch_size, -1, -1)  # shape: (batch_size, 14, 2)

    shunt_power = complex_mult_batch(complex_conj_batch(Yks_batch), v_mag_sq)
    
    # Pre-allocate branch flows dictionary with tensors
    branch_flows = {}
    
    # Calculate branch flows (vectorized)
    for idx, (i, j) in enumerate(branches):
        
        # Get complex voltage at both ends
        vi = voltage[:, i]  # shape: (batch_size, 2)
        vj = voltage[:, j]  # shape: (batch_size, 2)
        
        # First term calculations
        vi_mag_sq = torch.sum(vi**2, dim=-1, keepdim=True)  # shape: (batch_size, 1)
        vi_mag_sq = torch.cat([vi_mag_sq, torch.zeros_like(vi_mag_sq)], dim=-1)  # shape: (batch_size, 2)
        
        tij_mag_sq = torch.sum(Tij[idx]**2).unsqueeze(0)
        tij_mag_sq_tensor = torch.tensor([tij_mag_sq, 0.0], dtype=torch.float32).expand(batch_size, -1)
        
        vi_over_tij_sq = complex_div_batch(vi_mag_sq, tij_mag_sq_tensor)
        
        # Sum of branch admittance and charging admittance
        Y_total = torch.stack([
            Yij[idx, 0] + Yijc[idx, 0],
            Yij[idx, 1] + Yijc[idx, 1]
        ]).expand(batch_size, -1)
        
        term1 = complex_mult_batch(complex_conj_batch(Y_total), vi_over_tij_sq)
        
        # Second term calculations
        vivj = complex_mult_batch(vi, complex_conj_batch(vj))
        term2 = complex_mult_batch(
            complex_conj_batch(Yij[idx].expand(batch_size, -1)),
            complex_div_batch(vivj, Tij[idx].expand(batch_size, -1))
        )
        
        # Total branch flow Sij
        Sij = term1 - term2
        branch_flows[(i, j, idx)] = Sij
        
        # Reverse flow calculations
        vj_mag_sq = torch.sum(vj**2, dim=-1, keepdim=True)
        vj_mag_sq = torch.cat([vj_mag_sq, torch.zeros_like(vj_mag_sq)], dim=-1)
        
        term1_ji = complex_mult_batch(complex_conj_batch(Y_total), vj_mag_sq)
        vjvi = complex_mult_batch(complex_conj_batch(vi), vj)
        term2_ji = complex_mult_batch(
            complex_conj_batch(Yij[idx].expand(batch_size, -1)),
            complex_div_batch(vjvi, complex_conj_batch(Tij[idx].expand(batch_size, -1)))
        )
        
        Sji = term1_ji - term2_ji
        branch_flows[(j, i, idx)] = Sji
    
    
    # Aggregate generator power for each node (vectorized)
    for i in range(num_nodes):
        for index, (from_bus, to_bus) in enumerate(branches):
            if from_bus == i:
                generator_power[:, i] += branch_flows[(from_bus, to_bus, index)]
            if to_bus == i:
                generator_power[:, i] += branch_flows[(to_bus, from_bus, index)]
    
    generator_power += demand + shunt_power
    
    return generator_power, branch_flows

In [ ]:
def convert_to_complex_voltage(voltage_tensor):
    # Extract angle and magnitude
    voltage_angle = voltage_tensor[:,:,0:1]  # In radians
    voltage_magnitude = voltage_tensor[:,:,1:]
    
    # Calculate real and imaginary parts
    real_voltage = voltage_magnitude * torch.cos(voltage_angle)
    imaginary_voltage = voltage_magnitude * torch.sin(voltage_angle)
    
    return torch.concat((real_voltage,imaginary_voltage),dim=2)

In [ ]:
def convert_to_complex_rectangle(tensor_2d):
    # Extract angle and magnitude
    tensor_mag = tensor_2d[:,0:1]  # In radians
    tensor_angle = tensor_2d[:,1:]
    
    # Calculate real and imaginary parts
    real_tensor = tensor_mag * torch.cos(tensor_angle)
    imaginary_tensor = tensor_mag * torch.sin(tensor_angle)
    
    return torch.concat((real_tensor,imaginary_tensor),dim=1)

In [ ]:
conductance_susceptance = np.concatenate((edge_g, edge_b), axis=1)
conductance_susceptance = torch.tensor(conductance_susceptance).to(torch.float32)
charging_susceptance = torch.zeros_like(conductance_susceptance).to(torch.float32)
charging_susceptance[:,1:] =  edge_inputs[:,2:3].to('cpu')
Tij = edge_inputs[:,9:].to('cpu').to(torch.float32)
Tij[:grid_ac_line_features.shape[1],0:1] = 1.0
Tij_rec = convert_to_complex_rectangle(Tij)

In [ ]:
complex_v = convert_to_complex_voltage(v_predictions)
complex_v = complex_v.to(torch.float32)

In [ ]:
load_demand = load_demand.to(torch.float32)

In [ ]:
Yks = torch.zeros(grid_shunt[test_intersection].shape[0], system_size, grid_shunt[test_intersection].shape[-1]).to(torch.float32)

Yks[:,shunt_indices,:] = torch.tensor(grid_shunt[test_intersection])

Yks = Yks[:,:, [1, 0]]

In [ ]:
Yks.shape

In [ ]:
injection_balance,branch_flows = calculate_only_branch_flows(load_demand.to('cpu'),complex_v.to('cpu'),branch_list,Yks,conductance_susceptance,charging_susceptance,Tij_rec)

In [ ]:
def compute_optimality(test_inputs, test_outputs, test_objective):

    test_inputs = test_inputs.cpu()
    test_outputs = test_outputs.cpu()
    test_objective = test_objective.cpu()

    c2 = test_inputs[:,:,8:9]
    c1 = test_inputs[:,:,9:10]
    c0 = test_inputs[:,:,10:11]

    # Get relevant output dimensions (zero-indexed)
    p_gens = test_outputs[:,:,0:1] # select on Pgs for generators
  

    print('the shape of c2 is ', c2.shape)
    print('the shape of p_gen is ', p_gens.shape)
    
    # Compute node-wise metrics
    system_metrics = c2 * (p_gens ** 2) + c1 * p_gens + c0


    model_obj = torch.sum(system_metrics, dim=1)

    print('the shape of test objective is ', test_objective.shape)
    print('the shape of model objective is ', model_obj.shape)

    print(f'average model objective is {model_obj.mean()}')
    print(f'average IPOPT objective is {test_objective.mean()}')

    optimality_gap = (model_obj / test_objective) * 100

    
    return optimality_gap.mean()

In [ ]:
test_obj = torch.tensor(solution_objective_n1[test_intersection])
test_gen_inputs = torch.tensor(grid_generator[test_intersection])

In [ ]:
opt_gap = compute_optimality(test_gen_inputs, p_predictions, test_obj)

In [ ]:
print('optimality gap now is ', opt_gap)

In [ ]:
def calculate_angle_differences(angles, edges):
  
    # Ensure angles is a NumPy array
    angles = np.asarray(angles)
    
    # Initialize array to store angle differences
    angle_differences = np.zeros(len(edges), dtype=np.float32)
    
    # Calculate angle differences for each edge
    for i, (node1, node2) in enumerate(edges):
        angle_differences[i] = angles[node2] - angles[node1]
    
    return angle_differences

In [ ]:
##### time to evaluate constrain satisfactions

In [ ]:
predicted_angle_differences = np.zeros((v_predictions.shape[0], len(branch_list)))
predicted_angles = v_predictions[:,:,0]

for j in range(v_predictions.shape[0]):
    angle_differences = calculate_angle_differences(predicted_angles[j],branch_list)
    predicted_angle_differences[j] = angle_differences

In [ ]:
true_angle_differences = np.zeros((v_targets.shape[0], len(branch_list)))
true_angles = v_targets[:,:,0]

for j in range(v_targets.shape[0]):
    angle_differences = calculate_angle_differences(true_angles[j],branch_list)
    true_angle_differences[j] = angle_differences

In [ ]:
predicted_angle_differences = torch.tensor(predicted_angle_differences)
true_angle_differences = torch.tensor(true_angle_differences)

In [ ]:
# voltage angle difference bound 
angle_diff_upper = torch.full(predicted_angle_differences.shape, 0.5236)
angle_diff_lower = torch.full(predicted_angle_differences.shape, -0.5236)

# Calculate violations
lower_angle_violations = torch.clamp(angle_diff_lower - predicted_angle_differences, min=0)  # Positive if below lower bound
upper_angle_violations = torch.clamp(predicted_angle_differences - angle_diff_upper, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
angle_diff_violations = lower_angle_violations + upper_angle_violations

print('max voltage angle difference violation is : ',angle_diff_violations.max())
print('average voltage angle difference violation is : ',angle_diff_violations.mean())

In [ ]:
test_bus_inputs = torch.tensor(grid_bus[test_intersection])

In [ ]:
## voltage magnitude bound
vmin = test_bus_inputs[:,:,2:3].to('cpu')

vmax = test_bus_inputs[:,:,3:4].to('cpu')

lower_vmag_violation = torch.clamp(vmin - v_predictions[:,:,1:2], min=0)  # Positive if below lower bound

upper_vmag_violation = torch.clamp(v_predictions[:,:,1:2] - vmax, min=0)  # Positive if above upper bound

vmag_violations = lower_vmag_violation + upper_vmag_violation

print('max voltage magnitude violation is : ',vmag_violations.max())
print('average voltage magnitude violation is : ',vmag_violations.mean())

In [ ]:
test_generator_inputs = torch.tensor(grid_generator[test_intersection])

In [ ]:
# Gen active power bounds 
pmin = test_generator_inputs[:,:,2:3].to('cpu')

pmax = test_generator_inputs[:,:,3:4].to('cpu')


p_gens = p_predictions[:,:,0:1]


lower_pgen_violations = torch.clamp(pmin - p_gens, min=0)  # Positive if below lower bound
upper_pgen_violations = torch.clamp(p_gens - pmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
pgen_violations = lower_pgen_violations + upper_pgen_violations

print('max gen active power violation is : ',pgen_violations.max())
print('average active power violation is : ',pgen_violations.mean())

In [ ]:
# Gen reactive power bounds 
qmin = test_generator_inputs[:,:,5:6].to('cpu')

qmax = test_generator_inputs[:,:,6:7].to('cpu')


q_gens = p_predictions[:,:,1:2]


lower_qgen_violations = torch.clamp(qmin - q_gens, min=0)  # Positive if below lower bound
upper_qgen_violations = torch.clamp(q_gens - qmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
qgen_violations = lower_qgen_violations + upper_qgen_violations

print('max gen reactive power violation is : ',qgen_violations.max())
print('average reactive power violation is : ',qgen_violations.mean())

In [ ]:
forward_keys = [(i, j, index) for index, (i,j) in enumerate(branch_list)]
reverse_keys = [(j, i, index) for index, (i,j) in enumerate(branch_list)]

In [ ]:
len(forward_keys)

In [ ]:
len(reverse_keys)

In [ ]:
forward_branch_flows = {key: branch_flows[key] for key in forward_keys if key in branch_flows}
reverse_branch_flows = {key: branch_flows[key] for key in reverse_keys if key in branch_flows}

In [ ]:
forward_power_flows = [tensor.unsqueeze(dim=1) for tensor in forward_branch_flows.values()]

# Step 2: Concatenate tensors along the second axis (dim=1)
forward_power_flows = torch.cat(forward_power_flows, dim=1)

In [ ]:
reverse_power_flows = [tensor.unsqueeze(dim=1) for tensor in reverse_branch_flows.values()]

# Step 2: Concatenate tensors along the second axis (dim=1)
reverse_power_flows = torch.cat(reverse_power_flows, dim=1)

In [ ]:
# compute the power magnitudes

# Separate real and imaginary parts
def convert_to_power_magnitude(power_flow):
    
    real = power_flow[..., 0]  
    imag = power_flow[..., 1]  

    # Compute the magnitudes
    magnitudes = torch.sqrt(real**2 + imag**2) 


    result_tensor = magnitudes.unsqueeze(-1)

    return result_tensor

In [ ]:
forward_flow_magnitude = convert_to_power_magnitude(forward_power_flows)
reverse_flow_magnitude = convert_to_power_magnitude(reverse_power_flows)


In [ ]:
# Branch flow bounds in forward direction

long_term_line_rating = edge_inputs[:,6:7].to('cpu')

branch_flow_limit = long_term_line_rating.tile((p_predictions.shape[0],1,1))

forward_branch_flow = forward_flow_magnitude

forward_flow_violations = torch.clamp(forward_branch_flow - branch_flow_limit, min=0)  # Positive if above upper bound

print('max forward power flow violation is : ',forward_flow_violations.max())
print('average  forward power flow violation is : ',forward_flow_violations.mean())

In [ ]:
# Branch flow bounds in reverse direction

reverse_branch_flow = reverse_flow_magnitude

reverse_flow_violations = torch.clamp(reverse_flow_magnitude - branch_flow_limit, min=0)  # Positive if above upper bound

print('max reverse power flow violation is : ', reverse_flow_violations.max())
print('average reverse power flow violation is : ', reverse_flow_violations.mean())

In [ ]:
gen_power = torch.zeros(p_predictions.shape[0],system_size,2)
# gen_power[:,generator_indices,:] = p_targets
for i in range(p_predictions.shape[0]):
    gen_power[i, :, :].index_add_(0, torch.tensor(generator_indices), p_predictions[i, :, :])

In [ ]:
## Evaluate power balance constraint violations


real_power_balance_mismatches = injection_balance[:,:,0] - gen_power[:,:,0]


print('max active power balance mismatch is : ', real_power_balance_mismatches.max())
print('average active power balance mismatch is : ', real_power_balance_mismatches.mean())

In [ ]:
reactive_power_balance_mismatches = injection_balance[:,:,1] - gen_power[:,:,1]


print('max reactive power balance mismatch is : ', reactive_power_balance_mismatches.max())
print('average reactive power balance mismatch is : ', reactive_power_balance_mismatches.mean())

In [ ]:
#create table to save important metrics
columns=["metric", "value"]
model_metrics_table = wandb.Table(columns=columns)

In [ ]:
model_metrics_table.add_data("optimality gap", opt_gap)
model_metrics_table.add_data("max voltage angle difference violation", angle_diff_violations.max())
model_metrics_table.add_data("average voltage angle difference violation", angle_diff_violations.mean())
model_metrics_table.add_data("max voltage magnitude violation", vmag_violations.max())
model_metrics_table.add_data("average voltage magnitude violation", vmag_violations.mean())
model_metrics_table.add_data("max gen active power violation", pgen_violations.max())
model_metrics_table.add_data("average gen active power violation", pgen_violations.mean())
model_metrics_table.add_data("max gen reactive power violation", qgen_violations.max())
model_metrics_table.add_data("average gen reactive power violation", qgen_violations.mean())
model_metrics_table.add_data("max forward power flows violation", forward_flow_violations.max())
model_metrics_table.add_data("average forward power flows violation", forward_flow_violations.mean())
model_metrics_table.add_data("max reverse power flows violation", reverse_flow_violations.max())
model_metrics_table.add_data("average reverse power flows violation", reverse_flow_violations.mean())
model_metrics_table.add_data("max active power balance mismatch", real_power_balance_mismatches.max())
model_metrics_table.add_data("average active power balance mismatch", real_power_balance_mismatches.mean())
model_metrics_table.add_data("max reactive power balance mismatch", reactive_power_balance_mismatches.max())
model_metrics_table.add_data("average reactive power balance mismatch", reactive_power_balance_mismatches.mean())
wandb.log({"model_metrics_table" : model_metrics_table})

In [ ]:
#########################################################################################################
#### everything after here is to do power flows post processing ##################

In [ ]:
# def residual_function(x, Ybus):
#     V, theta, P, Q = torch.split(x, n_buses, dim=0)
#     v_complex = V * torch.exp(1j * theta)
#     i_complex = Ybus @ v_complex
#     s_complex = v_complex * torch.conj(i_complex)
#     residual_complex = (P + 1j * Q) - s_complex
#     residual = torch.vstack(
#         [torch.real(residual_complex), torch.imag(residual_complex)]
#     )  # shape (2*n_buses, 1)
#     return residual


In [ ]:
# def jacobian_function(x, Ybus):
#     jacobian = torch.autograd.functional.jacobian(
#         lambda variables: residual_function(variables, Ybus), x
#     )
#     return jacobian[:, 0, :, 0]

In [ ]:
# def newton_raphson_step(x, Ybus, update_indices):
#     residual = residual_function(x, Ybus)
#     jacobian = jacobian_function(x, Ybus)
#     delta_x = -torch.linalg.inv(jacobian[:, update_indices]) @ residual
#     x[update_indices, :] += delta_x
#     return x

In [ ]:
# def compute_update_indices(slack_bus, PV_indices, n_buses):
#     """
#     Compute the update indices for the power flow algorithm.
#     slack_bus: index of the slack bus
#     PV_indices: indices of the PV buses
#     n_buses: total number of buses in the system
#     """
#     update_V = [True] * n_buses
#     update_theta = [True] * n_buses
#     update_P = [False] * n_buses
#     update_Q = [False] * n_buses

#     for bus in range(n_buses):
#         if bus in slack_bus:
#             update_V[bus] = False
#             update_theta[bus] = False
#             update_P[bus] = True
#             update_Q[bus] = True
#         elif bus in PV_indices:
#             update_V[bus] = False
#             update_Q[bus] = True
#         else:
#             continue
#     update_indices = update_V + update_theta + update_P + update_Q
#     return update_indices


In [ ]:
# def prepare_variables(
#     voltage: torch.Tensor, # shape (14,2)
#     one_load: torch.Tensor, # shape (14,2)
#     power_input: torch.Tensor, # shape (14,2)
#     bus_types: torch.Tensor # shape flat tensor
# ):

#     num_buses, _ = voltage.shape
        
#     # get magnitudes and angles
#     V = torch.norm(voltage, dim=1)
#     theta = torch.atan2(voltage[:, 1], voltage[:, 0])

#     P_initial = power_input[:,0] - one_load[:,0]
#     Q_initial = power_input[:,1] - one_load[:,1]

#     slack_index = torch.where((bus_types == 3))[0]
#     PV_indices = torch.where(bus_types == 2)[0]

#     V = V.reshape(-1,1)
#     theta = theta.reshape(-1,1)
#     P_initial = P_initial.reshape(-1,1)
#     Q_initial = Q_initial.reshape(-1,1)
    
#     return V, theta, P_initial, Q_initial, slack_index, PV_indices


In [ ]:
# def calculate_Ybus_with_transformers(num_buses, branches, Yij, Yijc, Yks, Tij):
#     Ybus = torch.zeros((num_buses, num_buses, 2), dtype=torch.float32)
    
#     # For convenience, convert to complex representation
#     Y_complex = torch.zeros((num_buses, num_buses), dtype=torch.complex64)
    
#     # Initialize diagonal elements with shunt admittances
#     for i in range(num_buses):
#         Y_complex[i, i] = complex(Yks[i, 0], Yks[i, 1])
    
#     # Process each branch
#     for idx, (i, j) in enumerate(branches):
#         # Get branch series admittance
#         y_series = complex(Yij[idx, 0], Yij[idx, 1])
#         # Get branch shunt admittance
#         y_shunt = complex(Yijc[idx, 0], Yijc[idx, 1])
#         # Get transformer ratio
#         t = complex(Tij[idx, 0], Tij[idx, 1])
        
#         # Off-diagonal elements
#         Y_complex[i, j] -= y_series / t
#         Y_complex[j, i] -= y_series / t.conjugate()
        
#         # Diagonal elements
#         Y_complex[i, i] += y_series / (t * t.conjugate()) + y_shunt/2
#         Y_complex[j, j] += y_series + y_shunt/2
    
#     # Convert back to your tensor format
#     for i in range(num_buses):
#         for j in range(num_buses):
#             Ybus[i, j, 0] = Y_complex[i, j].real
#             Ybus[i, j, 1] = Y_complex[i, j].imag
    
#     return Ybus

In [ ]:
# Ybus = calculate_Ybus_with_transformers(system_size,branch_list,conductance_susceptance,charging_susceptance,Yks[0],Tij_rec)

In [ ]:
# Ybus_complex = Ybus[:,:,0] + (1j * Ybus[:,:,1])
# Ybus = Ybus_complex

In [ ]:
# complex_v_copy = complex_v.detach().clone()

In [ ]:
# generator_Ps = gen_power[:,:,0].unsqueeze(-1)
# generator_Qs = gen_power[:,:,1].unsqueeze(-1)

In [ ]:
# power_inputs = torch.cat((generator_Ps,generator_Qs),dim=2)

In [ ]:
# bus_types = torch.tensor(grid_bus[0,:,1], dtype =int)
# n_buses = len(bus_types)

In [ ]:
# power_inputs = power_inputs.to(torch.float32).to(device)
# conductance_susceptance = conductance_susceptance.to(torch.float32).to(device)
# charging_susceptance = charging_susceptance.to(torch.float32).to(device)
# shunt_input = Yks.to(torch.float32).to(device)
# Tij_rec = Tij_rec.to(torch.float32).to(device)
# pre_power_flow_voltage = complex_v_copy.to(torch.float32).to(device) #### changes here
# load_input = load_demand.to(torch.float32).to(device) #### changes here
# bus_types = bus_types.to(torch.float32).to(device)
# Ybus = Ybus.to(torch.complex64).to(device)

In [ ]:
# post_flow_powers = torch.zeros_like(power_inputs).to(device)
# post_flow_voltages = torch.zeros_like(pre_power_flow_voltage).to(device)

In [ ]:
# start_powerflow_time = time.time()

# for k in range(post_flow_powers.shape[0]):
#     V_initial, theta_initial, P_initial, Q_initial, slack_index, PV_indices = prepare_variables(pre_power_flow_voltage[k],load_input[k],power_inputs[k],bus_types)
#     x_initial = torch.cat([V_initial,theta_initial,P_initial,Q_initial])

#     x_current = x_initial.clone()
#     update_indices = compute_update_indices(slack_index, PV_indices, n_buses)

#     for ii in range(10):
#         x_new = newton_raphson_step(x_current, Ybus, update_indices)
#         x_current = x_new.clone()

#     if (torch.norm(residual_function(x_current, Ybus))/ torch.sqrt(torch.tensor(2.0 * n_buses)) > 1e-3): 
#         print(f"power flows for instance {k} did not converge")
#         print(f"the normalized residual after power flow is still {torch.norm(residual_function(x_current, Ybus))/ torch.sqrt(torch.tensor(2.0 * n_buses))}")
    
#     V_solution, theta_solution, P_solution, Q_solution = torch.split(x_current, n_buses, dim=0)
#     post_flow_Pgen = P_solution + load_input[k][:,0:1]
#     post_flow_Qgen = Q_solution + load_input[k][:,1:2]
#     post_flow_powers[k,:,0:1] = post_flow_Pgen
#     post_flow_powers[k,:,1:2] = post_flow_Qgen
#     post_flow_voltages[k,:,0:1] = V_solution
#     post_flow_voltages[k,:,1:2] = theta_solution
    
# end_powerflow_time = time.time()
# total_powerflow_time = end_powerflow_time - start_powerflow_time
# print(f"Time taken on power flow is: {total_powerflow_time} second")

In [ ]:
# post_flow_powers = post_flow_powers.to('cpu')
# post_flow_voltages = post_flow_voltages.to('cpu')

In [ ]:
# def batch_newton_raphson_step(V, theta, P, Q, Ybus, update_indices_mask, batch_size, n_buses):
#     # Initialize new values
#     V_new = V.clone()
#     theta_new = theta.clone()
#     P_new = P.clone()
#     Q_new = Q.clone()
    
#     # Process each batch item
#     for b in range(batch_size):
#         # Construct x for this batch
#         x_b = torch.cat([
#             V[b].reshape(-1), 
#             theta[b].reshape(-1), 
#             P[b].reshape(-1), 
#             Q[b].reshape(-1)
#         ], dim=0)
        
#         # Get update indices for this batch
#         update_indices_b = torch.where(update_indices_mask[b])[0]
        
#         # Define the residual function for a single batch
#         def residual_func(x):
#             V_temp, theta_temp, P_temp, Q_temp = torch.split(x, n_buses)
#             V_temp = V_temp.reshape(-1, 1)
#             theta_temp = theta_temp.reshape(-1, 1)
#             P_temp = P_temp.reshape(-1, 1)
#             Q_temp = Q_temp.reshape(-1, 1)
#             return vectorized_residual_function(
#                 V_temp.unsqueeze(0), 
#                 theta_temp.unsqueeze(0), 
#                 P_temp.unsqueeze(0), 
#                 Q_temp.unsqueeze(0), 
#                 Ybus, 
#                 1
#             )[0, :, 0]  # Extract [2*n_buses] vector
        
#         # Compute residual for current state
#         residual_b = residual_func(x_b)
        
#         # Compute Jacobian
#         jacobian = torch.autograd.functional.jacobian(residual_func, x_b)
        
#         # Extract the submatrix corresponding to the update indices
#         J_reduced = jacobian[:, update_indices_b]
#         r_reduced = residual_b
        
#         # Solve the linear system
#         delta_x = -torch.linalg.solve(J_reduced, r_reduced)
        
#         # Update state variables
#         x_b_new = x_b.clone()
#         x_b_new[update_indices_b] += delta_x
        
#         # Split updated state back into components
#         V_b_new, theta_b_new, P_b_new, Q_b_new = torch.split(x_b_new, n_buses)
        
#         # Store updated values
#         V_new[b] = V_b_new.reshape(-1, 1)
#         theta_new[b] = theta_b_new.reshape(-1, 1)
#         P_new[b] = P_b_new.reshape(-1, 1)
#         Q_new[b] = Q_b_new.reshape(-1, 1)
    
#     return V_new, theta_new, P_new, Q_new

# def vectorized_residual_function(V, theta, P, Q, Ybus, batch_size):
#     # Reshape tensors to ensure consistent dimensions
#     V = V.reshape(batch_size, -1, 1)        # [batch_size, n_buses, 1]
#     theta = theta.reshape(batch_size, -1, 1) # [batch_size, n_buses, 1]
#     P = P.reshape(batch_size, -1, 1)        # [batch_size, n_buses, 1]
#     Q = Q.reshape(batch_size, -1, 1)        # [batch_size, n_buses, 1]
    
#     v_complex = V * torch.exp(1j * theta)   # [batch_size, n_buses, 1]
    
#     # Process one batch at a time for matrix multiplication
#     i_complex = torch.zeros_like(v_complex, dtype=torch.complex64)
#     for b in range(batch_size):
#         i_complex[b] = Ybus @ v_complex[b]  # [n_buses, 1]
    
#     s_complex = v_complex * torch.conj(i_complex)  # [batch_size, n_buses, 1]
#     residual_complex = (P + 1j * Q) - s_complex    # [batch_size, n_buses, 1]
    
#     # Stack real and imaginary parts
#     residual = torch.cat(
#         [torch.real(residual_complex), torch.imag(residual_complex)], dim=1
#     )  # [batch_size, 2*n_buses, 1]
    
#     return residual

# def vectorized_power_flow(pre_power_flow_voltage, load_input, power_inputs, bus_types, Ybus, max_iterations=10, tolerance=1e-3):
#     batch_size, n_buses, _ = pre_power_flow_voltage.shape
#     device = pre_power_flow_voltage.device
    
#     # Prepare variables for all batches
#     V_all = torch.zeros((batch_size, n_buses, 1), dtype=torch.float32, device=device)
#     theta_all = torch.zeros((batch_size, n_buses, 1), dtype=torch.float32, device=device)
#     P_all = torch.zeros((batch_size, n_buses, 1), dtype=torch.float32, device=device)
#     Q_all = torch.zeros((batch_size, n_buses, 1), dtype=torch.float32, device=device)
    
#     # Create update indices mask for all batches
#     update_indices_mask = torch.zeros((batch_size, 4 * n_buses), dtype=torch.bool, device=device)
    
#     # Find slack and PV buses
#     slack_indices = torch.where(bus_types == 3)[0]
#     PV_indices = torch.where(bus_types == 2)[0]
    
#     # Prepare initial values and masks for all batches
#     for b in range(batch_size):
#         # Calculate V and theta from complex voltage
#         V = torch.norm(pre_power_flow_voltage[b], dim=1, keepdim=True)
#         theta = torch.atan2(pre_power_flow_voltage[b, :, 1:2], pre_power_flow_voltage[b, :, 0:1])
        
#         # Initial P and Q
#         P = power_inputs[b, :, 0:1] - load_input[b, :, 0:1]
#         Q = power_inputs[b, :, 1:2] - load_input[b, :, 1:2]
        
#         V_all[b] = V
#         theta_all[b] = theta
#         P_all[b] = P
#         Q_all[b] = Q
        
#         # Create update mask for this batch
#         update_V = torch.ones(n_buses, dtype=torch.bool, device=device)
#         update_theta = torch.ones(n_buses, dtype=torch.bool, device=device)
#         update_P = torch.zeros(n_buses, dtype=torch.bool, device=device)
#         update_Q = torch.zeros(n_buses, dtype=torch.bool, device=device)
        
#         # Set update flags based on bus types
#         for idx in slack_indices:
#             update_V[idx] = False
#             update_theta[idx] = False
#             update_P[idx] = True
#             update_Q[idx] = True
        
#         for idx in PV_indices:
#             update_V[idx] = False
#             update_Q[idx] = True
        
#         # Concatenate all update flags
#         batch_update_indices = torch.cat([update_V, update_theta, update_P, update_Q])
#         update_indices_mask[b] = batch_update_indices
    
#     # Newton-Raphson iterations
#     for iter_idx in range(max_iterations):
#         V_all, theta_all, P_all, Q_all = batch_newton_raphson_step(
#             V_all, theta_all, P_all, Q_all, Ybus, update_indices_mask, batch_size, n_buses
#         )
        
#         # Check convergence (every few iterations to save computation)
#         if iter_idx % 2 == 0 or iter_idx == max_iterations - 1:
#             max_residuals = []
#             for b in range(batch_size):
#                 residual = vectorized_residual_function(
#                     V_all[b:b+1], theta_all[b:b+1], P_all[b:b+1], Q_all[b:b+1], Ybus, 1
#                 )
#                 normalized_residual = torch.norm(residual) / torch.sqrt(torch.tensor(2.0 * n_buses, device=device))
#                 max_residuals.append(normalized_residual)
            
#             # If all converged, we can stop early
#             if all(r < tolerance for r in max_residuals):
#                 # print(f"Converged after {iter_idx+1} iterations")
#                 break
    
#     # Check convergence for each batch
#     non_converged = 0
#     for b in range(batch_size):
#         residual = vectorized_residual_function(
#             V_all[b:b+1], theta_all[b:b+1], P_all[b:b+1], Q_all[b:b+1], Ybus, 1
#         )
#         normalized_residual = torch.norm(residual) / torch.sqrt(torch.tensor(2.0 * n_buses, device=device))
#         if normalized_residual > tolerance:
#             non_converged += 1
#             if non_converged <= 5:  # Limit output to avoid flooding console
#                 print(f"Power flow for instance {b} did not converge")
#                 print(f"The normalized residual after power flow is still {normalized_residual.item()}")
    
#     if non_converged > 5:
#         print(f"...and {non_converged-5} more cases didn't converge")
    
#     # Prepare output
#     post_flow_powers = torch.zeros_like(power_inputs)
#     post_flow_voltages = torch.zeros_like(pre_power_flow_voltage)
    
#     for b in range(batch_size):
#         post_flow_powers[b, :, 0:1] = P_all[b] + load_input[b, :, 0:1]
#         post_flow_powers[b, :, 1:2] = Q_all[b] + load_input[b, :, 1:2]
        
#         # Convert V and theta back to complex voltage
#         post_flow_voltages[b, :, 0:1] = V_all[b]
#         post_flow_voltages[b, :, 1:2] = theta_all[b]
    
#     return post_flow_voltages, post_flow_powers

# # Main execution
# def run_vectorized_power_flow(pre_power_flow_voltage, load_input, power_inputs, bus_types, Ybus, total_cases=15000):
#     # Start timing
#     start_powerflow_time = time.time()
#     device = pre_power_flow_voltage.device
    
#     # Initialize output tensors
#     n_buses = pre_power_flow_voltage.shape[1]
#     post_flow_voltages = torch.zeros_like(pre_power_flow_voltage)
#     post_flow_powers = torch.zeros_like(power_inputs)
    
#     # Set batch size based on GPU memory
#     # For 2000-bus system, start with smaller batches
#     batch_size = 16  # Adjust based on GPU memory and system size
    
#     for batch_start in range(0, total_cases, batch_size):
#         batch_end = min(batch_start + batch_size, total_cases)
#         current_batch_size = batch_end - batch_start
        
#         print(f"Processing batch {batch_start//batch_size + 1}/{(total_cases+batch_size-1)//batch_size}")
        
#         # Process current batch
#         batch_voltages, batch_powers = vectorized_power_flow(
#             pre_power_flow_voltage[batch_start:batch_end],
#             load_input[batch_start:batch_end],
#             power_inputs[batch_start:batch_end],
#             bus_types,
#             Ybus,
#             max_iterations=10,
#             tolerance=1e-3
#         )
        
#         # Store results
#         post_flow_voltages[batch_start:batch_end] = batch_voltages
#         post_flow_powers[batch_start:batch_end] = batch_powers
    
#     end_powerflow_time = time.time()
#     total_powerflow_time = end_powerflow_time - start_powerflow_time
#     print(f"Time taken on power flow is: {total_powerflow_time} seconds")
#     print(f"Average time per case: {total_powerflow_time/total_cases} seconds")
    
#     return post_flow_voltages, post_flow_powers


In [ ]:
# # Execute the vectorized power flow
# post_flow_voltages, post_flow_powers = run_vectorized_power_flow(pre_power_flow_voltage, load_input, power_inputs, bus_types, Ybus)

In [ ]:
# post_flow_voltages = post_flow_voltages[:,:, [1, 0]]

In [ ]:
# post_flow_powers = post_flow_powers.to('cpu')
# post_flow_voltages = post_flow_voltages.to('cpu')

In [ ]:
# torch.save(post_flow_voltages, f"{system_size}_HybridGNN_post_powerflow_voltages.pt")
# torch.save(post_flow_powers, f"{system_size}_HybridGNN_post_powerflow_powers.pt")

In [ ]:
# time to re-evaluate constraint satisfactions

In [ ]:
# new_angle_differences = torch.zeros((post_flow_voltages.shape[0], len(branch_list)))
# new_angles = post_flow_voltages[:,:,0]

# for j in range(post_flow_voltages.shape[0]):
#     one_angle_differences = calculate_angle_differences(new_angles[j],branch_list)
#     one_angle_differences = torch.tensor(one_angle_differences)
#     new_angle_differences[j] = one_angle_differences

In [ ]:
# # voltage angle difference bound 
# new_angle_diff_upper = torch.full(new_angle_differences.shape, 0.5236)
# new_angle_diff_lower = torch.full(new_angle_differences.shape, -0.5236)

# # Calculate violations
# new_lower_angle_violations = torch.clamp(new_angle_diff_lower - new_angle_differences, min=0)  # Positive if below lower bound
# new_upper_angle_violations = torch.clamp(new_angle_differences - new_angle_diff_upper, min=0)  # Positive if above upper bound

# # Combine violations into a single tensor
# new_angle_diff_violations = new_lower_angle_violations + new_upper_angle_violations

# print('max voltage angle difference violation post powerflow is : ',new_angle_diff_violations.max())
# print('average voltage angle difference violation post powerflow is : ',new_angle_diff_violations.mean())

In [ ]:
# ## voltage magnitude bound

# vmin = test_bus_inputs[:,:,2:3].to('cpu')

# vmax = test_bus_inputs[:,:,3:4].to('cpu')

# new_lower_vmag_violation = torch.clamp(vmin - post_flow_voltages[:,:,1:], min=0)  # Positive if below lower bound

# new_upper_vmag_violation = torch.clamp(post_flow_voltages[:,:,1:] - vmax, min=0)  # Positive if above upper bound

# new_vmag_violations = new_lower_vmag_violation + new_upper_vmag_violation

# print('max voltage magnitude violation  post powerflow is  : ',new_vmag_violations.max())
# print('average voltage magnitude violation  post powerflow is : ',new_vmag_violations.mean())

In [ ]:
# # Gen active power bounds 
# new_pmin = test_generator_inputs[:,:,2:3].to('cpu')

# new_pmax = test_generator_inputs[:,:,3:4].to('cpu')


# pf_pgens= post_flow_powers[:,generator_indices,0:1]

# new_lower_pgen_violations = torch.clamp(new_pmin - pf_pgens, min=0)  # Positive if below lower bound
# new_upper_pgen_violations = torch.clamp(pf_pgens - new_pmax, min=0)  # Positive if above upper bound

# # Combine violations into a single tensor
# new_pgen_violations = new_lower_pgen_violations + new_upper_pgen_violations

# print('max gen active power violation post power flow is : ',new_pgen_violations.max() )
# print('average active power violation post power flow is : ',new_pgen_violations.mean() )

In [ ]:
# # Gen reactive power bounds 
# new_qmin = test_generator_inputs[:,:,6:7].to('cpu')

# new_qmax = test_generator_inputs[:,:,7:8].to('cpu')


# pf_qgens= post_flow_powers[:,generator_indices,1:2]

# new_lower_qgen_violations = torch.clamp(new_qmin - pf_qgens, min=0)  # Positive if below lower bound
# new_upper_qgen_violations = torch.clamp(pf_qgens - new_qmax, min=0)  # Positive if above upper bound

# # Combine violations into a single tensor
# new_qgen_violations = new_lower_qgen_violations + new_upper_qgen_violations

# print('max gen reactive power violation post power flow is : ', new_qgen_violations.max())
# print('average reactive power violation post power flow is : ', new_qgen_violations.mean())

In [ ]:
# new_complex_v = convert_to_complex_voltage(post_flow_voltages)
# new_complex_v = new_complex_v.to(device)

In [ ]:
# new_injection_balance,new_branch_flows = calculate_only_branch_flows(load_input.to('cpu'),new_complex_v.to('cpu'),branch_list,shunt_input.to('cpu'),conductance_susceptance.to('cpu'),charging_susceptance.to('cpu'),Tij_rec.to('cpu'))

In [ ]:
# new_forward_branch_flows = {key: new_branch_flows[key] for key in forward_keys if key in new_branch_flows}
# new_reverse_branch_flows = {key: new_branch_flows[key] for key in reverse_keys if key in new_branch_flows}

In [ ]:
# new_forward_power_flows = [tensor.unsqueeze(dim=1) for tensor in new_forward_branch_flows.values()]

# # Step 2: Concatenate tensors along the second axis (dim=1)
# new_forward_power_flows = torch.cat(new_forward_power_flows, dim=1)

In [ ]:
# new_reverse_power_flows = [tensor.unsqueeze(dim=1) for tensor in new_reverse_branch_flows.values()]


# # Step 2: Concatenate tensors along the second axis (dim=1)
# new_reverse_power_flows = torch.cat(new_reverse_power_flows, dim=1)

In [ ]:
# new_forward_flow_magnitude = convert_to_power_magnitude(new_forward_power_flows)
# new_reverse_flow_magnitude = convert_to_power_magnitude(new_reverse_power_flows)


In [ ]:
# # Branch flow bounds in forward direction

# long_term_line_rating = edge_inputs[:,6:7].to('cpu')

# branch_flow_limit = long_term_line_rating.tile((p_predictions.shape[0],1,1))

# new_forward_branch_flow = new_forward_flow_magnitude

# new_forward_flow_violations = torch.clamp(new_forward_branch_flow - branch_flow_limit, min=0)  # Positive if above upper bound

# print('max forward power flow violation post power flow is : ',new_forward_flow_violations.max() )
# print('average  forward power flow violation post power flow is : ',new_forward_flow_violations.mean() )

In [ ]:
# # Branch flow bounds in reverse direction

# new_reverse_branch_flow = new_reverse_flow_magnitude

# new_reverse_flow_violations = torch.clamp(new_reverse_flow_magnitude - branch_flow_limit, min=0)  # Positive if above upper bound

# print('max reverse power flow violation post power flow is : ', new_reverse_flow_violations.max() )
# print('average reverse power flow violation post power flow is : ', new_reverse_flow_violations.mean() )

In [ ]:
# ## Evaluate power balance constraint violations
# pf_real_power_balance_mismatches = new_injection_balance[:,:,0:1] - post_flow_powers[:,:,0:1]

# print('max active power balance mismatch post power flow is : ', pf_real_power_balance_mismatches.max() )
# print('average active power balance mismatch post power flow is : ', pf_real_power_balance_mismatches.mean())

In [ ]:
# pf_reactive_power_balance_mismatches = injection_balance[:,:,1:] - post_flow_powers[:,:,1:]


# print('max reactive power balance mismatch post power flow is : ', pf_reactive_power_balance_mismatches.max())
# print('average reactive power balance mismatch post power flow is : ', pf_reactive_power_balance_mismatches.mean())

In [ ]:
### last thing lelft to do is compute post power flow optimality gap

In [ ]:
# def compute_post_powerflow_optimality(test_inputs, test_outputs, test_objective):

#     test_inputs = test_inputs.cpu()
#     test_outputs = test_outputs.cpu()
#     test_objective = test_objective.cpu()

#     c2 = test_inputs[:,:,8:9]
#     c1 = test_inputs[:,:,9:10]
#     c0 = test_inputs[:,:,10:11]

#     # Get relevant output dimensions (zero-indexed)
#     p_gens = test_outputs[:,:,0:1] # select on Pgs for generators
  

#     print('the shape of c2 is ', c2.shape)
#     print('the shape of p_gen is ', p_gens.shape)
    
#     # Compute node-wise metrics
#     system_metrics = c2 * (p_gens ** 2) + c1 * p_gens + c0


#     model_obj = torch.sum(system_metrics, dim=(1,2))

#     print('the shape of test objective is ', test_objective.shape)
#     print('the shape of model objective is ', model_obj.shape)

#     print(f'average model objective is {model_obj.mean()}')
#     print(f'average IPOPT objective is {test_objective.mean()}')

#     optimality_gap = (model_obj / test_objective) * 100

    
#     return optimality_gap.mean()

In [ ]:
# active_gen_powers = post_flow_powers[:,:,0].unsqueeze(-1)
# active_gen_powers =  active_gen_powers[:,generator_indices].to(device)

In [ ]:
# pf_test_opt_gap = compute_post_powerflow_optimality(test_gen_inputs, active_gen_powers, test_obj)

In [ ]:
# print(f'The average optimality gap after power flow is: {pf_test_opt_gap:.5f}')

In [ ]:
# #create table to save important metrics
# columns=["metric", "value"]
# post_powerflow_metrics_table = wandb.Table(columns=columns)

In [ ]:
# post_powerflow_metrics_table.add_data("optimality gap", pf_test_opt_gap)
# post_powerflow_metrics_table.add_data("max voltage angle difference violation", new_angle_diff_violations.max())
# post_powerflow_metrics_table.add_data("average voltage angle difference violation", new_angle_diff_violations.mean())
# post_powerflow_metrics_table.add_data("max voltage magnitude violation", new_vmag_violations.max())
# post_powerflow_metrics_table.add_data("average voltage magnitude violation", new_vmag_violations.mean())
# post_powerflow_metrics_table.add_data("max gen active power violation", new_pgen_violations.max())
# post_powerflow_metrics_table.add_data("average gen active power violation", new_pgen_violations.mean())
# post_powerflow_metrics_table.add_data("max gen reactive power violation", new_qgen_violations.max())
# post_powerflow_metrics_table.add_data("average gen reactive power violation", new_qgen_violations.mean())
# post_powerflow_metrics_table.add_data("max forward power flows violation", new_forward_flow_violations.max())
# post_powerflow_metrics_table.add_data("average forward power flows violation", new_forward_flow_violations.mean())
# post_powerflow_metrics_table.add_data("max reverse power flows violation", new_reverse_flow_violations.max())
# post_powerflow_metrics_table.add_data("average reverse power flows violation", new_reverse_flow_violations.mean())
# post_powerflow_metrics_table.add_data("max active power balance mismatch", pf_real_power_balance_mismatches.max())
# post_powerflow_metrics_table.add_data("average active power balance mismatch", pf_real_power_balance_mismatches.mean())
# post_powerflow_metrics_table.add_data("max reactive power balance mismatch", pf_reactive_power_balance_mismatches.max())
# post_powerflow_metrics_table.add_data("average reactive power balance mismatch", pf_reactive_power_balance_mismatches.mean())
# wandb.log({"post_powerflow_metrics_table" : post_powerflow_metrics_table})

In [ ]:
wandb.finish()

In [ ]:
torch.cuda.empty_cache()
gc.collect()